# Teknik Feature Engineering

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_wine

load data dari scikit learn

In [2]:
# Memuat dataset Wine Quality
data = load_wine()
X, y = data.data, data.target

In [3]:
# Mengubah menjadi DataFrame untuk analisis yang lebih mudah
df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y
df

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0,2
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0,2
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0,2
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0,2


In [4]:
# Pembagian data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Filter Methods
Filter methods adalah teknik untuk menilai relevansi fitur secara independen dari model yang akan digunakan. Metode ini menggunakan statistik untuk memilih fitur tanpa melibatkan algoritma machine learning. Hal ini membuat proses pemilihan fitur dengan lebih cepat dan efisien, terutama untuk dataset besar      
- Kekurangan
    - Karena beroperasi secara independen, filter methods mungkin melewatkan interaksi data yang mungkin penting untuk prediksi.
    - Anda perlu melakukan perhitungan metriks yang tepat dan sesuai dengan algoritma yang akan digunakan. Hal ini karena memilih metrik yang sesuai untuk data dan tugas Anda sangat penting untuk kinerja yang optimal.
- Kelebihan
    - Metode filter tidak membutuhkan daya komputasi yang besar sehingga ideal untuk set data yang besar dengan kondisi keterbatasan hardware.
    - Metode-metode ini sering kali sudah ada di dalam library machine learning yang populer sehingga akan mempermudah pekerjaan Anda sebagai machine learning engineer.
    - Metode filter dapat digunakan dengan semua jenis model machine learning sehingga Anda tidak perlu takut salah. Namun, hal ini bukan berarti Anda dapat melakukan konsep All in One Methods, ya.

In [ ]:
# ------------------- Filter Methods -------------------
# Menggunakan SelectKBest
filter_selector = SelectKBest(score_func=chi2, k=2)  # Memilih 2 fitur terbaik
X_train_filter = filter_selector.fit_transform(X_train, y_train)
X_test_filter = filter_selector.transform(X_test)
 
print("Fitur yang dipilih dengan Filter Methods:", filter_selector.get_support(indices=True))

Fitur yang dipilih dengan Filter Methods: [ 9 12]


: 

SelectKBest
SelectKBest adalah salah satu metode dari filter methods yang umum digunakan untuk memilih fitur terbaik berdasarkan skor statistik tertentu.
        - Parameter score_func=chi2 berarti metode ini menggunakan Chi-squared (χ²) sebagai fungsi skor. Chi-squared adalah metode uji statistik yang digunakan untuk mengukur independensi antara dua variabel kategorikal.
        - Parameter k=2 artinya kita hanya akan memilih 2 fitur terbaik dari semua fitur yang ada di dataset. Dengan kata lain, ini akan menyeleksi dua fitur dengan skor tertinggi berdasarkan hasil uji Chi-squared.
        Kedua parameter ini bisa Anda sesuaikan dengan studi kasus dan trial and error ya. 

fit_transform(X_train, y_train)
fit(X_train, y_train) merupakan metode yang akan menghitung skor Chi-squared untuk setiap fitur pada data pelatihan (X_train) terhadap target (y_train).
transform(X_train) merupakan fungsi yang bertugas untuk menyalin data dari X_train, tetapi pada kasus ini hanya memiliki fitur yang tersimpan pada filter_selector.
Hasilnya adalah sebuah dataset pelatihan baru (X_train_filter) yang hanya mengandung dua fitur terbaik hasil perhitungan Chi-squared.

transform(X_test)
- Fungsi transform(X_test) digunakan untuk mentransformasikan dataset uji (X_test) dengan memilih fitur yang sama yang dipilih dari dataset pelatihan.
get_support(indices=True)
- Fungsi get_support(indices=True) digunakan untuk mendapatkan indeks dari fitur-fitur yang dipilih. Dalam hal ini, fitur yang dipilih adalah dua fitur dengan skor tertinggi berdasarkan uji Chi-squared.
= Fungsi ini akan mencetak indeks fitur yang dipilih dari dataset awal yang bisa digunakan untuk memahami fitur mana yang dipertahankan setelah proses seleksi. Sehingga, output akhir dari kode ini merupakan dataset yang sudah difilter dan informasi angka indeks dari fitur yang dipilih.


## Wrapper Methods
Wrapper methods bertugas untuk mengevaluasi subset fitur berdasarkan kinerja model machine learning. Metode ini lebih berat secara komputasional tetapi sering kali memberikan hasil yang lebih baik karena mempertimbangkan interaksi antara fitur.

Seperti yang dapat Anda lihat pada gambar di atas, metode ini akan mengevaluasi subset fitur berdasarkan kinerja model machine learning yang dipilih. Metode ini secara iteratif akan menambah atau menghapus fitur dengan tujuan untuk menemukan kombinasi yang menghasilkan kinerja model terbaik. Dalam penggunaannya Anda dapat menggunakan beberapa teknik untuk memaksimalkan proses feature selection.

Salah satu teknik yang umum digunakan pada metode ini adalah Recursive Feature Elimination (RFE) di mana proses iteratif akan menghapus fitur yang kurang penting dengan membangun model secara berulang. Metode lain yaitu forward selection dan backward elimination yang berfokus pada penambahan atau penghapusan fitur berdasarkan performa model dengan melakukan evaluasi menggunakan teknik cross-validation. Last but not least, untuk dataset kecil, exhaustive feature selection dapat dilakukan meskipun metode ini berat secara komputasional karena mencakup semua kombinasi fitur.

Metode ini memiliki beberapa kekurangan dan kelebihan seperti berikut.

Kekurangan
- Mengevaluasi kombinasi fitur yang berbeda dapat memakan waktu dan sumber daya yang besar terutama untuk data dengan ukuran besar.
- Menyesuaikan fitur pada model tertentu dapat menyebabkan model overfitting yaitu model yang memiliki kinerja buruk pada data yang yang belum pernah dilihat.
Kelebihan
- Wrapper methods secara langsung mempertimbangkan bagaimana fitur memengaruhi model sehingga berpotensi menghasilkan performa yang lebih baik dibandingkan dengan metode filter.
- Metode-metode ini dapat diadaptasi ke berbagai jenis model dan metriks evaluasi.